# Flux 01 — scFEA demo / proof-of-concept

Cilj: empiricno preveriti izvedljivost flux pipeline (TRIM-Flux Var 2) na MAJHNEM vzorcu, PREDEN gradimo poln pipeline.

Orodje: **scFEA** (single-cell Flux Estimation Analysis, GNN, GPU). Izbrano po raziskavi izvedljivosti:
- Compass ~30 min/vzorec -> per-cell na 147k neizvedljivo
- scFEA per-cell, GPU-pospesen, dropout-robusten (korelacija >0.85 pri simuliranem dropoutu)
- ~168 metabolnih modulov za cloveka (module_gene_m168)

Kaj ta demo izmeri:
1. namestitev scFEA + odvisnosti
2. format/orientacija vhoda (scFEA hoce GENE x CELICE; nasa data_rna je CELICE x GENI -> transponiraj)
3. ali so geni IMENA (scFEA rabi gene simbole) ali Ensembl ID
4. **cas izracuna** na vzorcu (npr. 2000 celic) -> ekstrapolacija na 147k
5. oblika izhoda (flux: CELICE x ~168 modulov)


## 0. Mount + namestitev scFEA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Namesti scFEA iz GitHub + odvisnosti
%cd /content
!git clone https://github.com/changwn/scFEA.git
%cd /content/scFEA

# scFEA importa 'magic' (MAGIC imputacija) na vrhu skripte -> NUJNO nalozen,
# tudi pri --sc_imputation False. Namestimo BREZ deps (sicer vlece pandas 2.0.3 iz vira -> build pade).
!pip install -q --no-deps magic-impute graphtools scprep s_gd2 pygsp Deprecated tasklogger wrapt
import magic  # preveri da import deluje
print('magic OK:', magic.__version__ if hasattr(magic,"__version__") else 'nalozen')

# PATCH 1: pandas 2.0 — DataFrame.append() odstranjen. *Df.append( -> *Df._append(
!sed -i -E 's/([A-Za-z_]*Df)\.append\(/\1._append(/g' src/scFEA.py
# PATCH 2: torch — .numpy() na GPU tenzorju pade. .detach().numpy() -> .detach().cpu().numpy()
!sed -i 's/\.detach()\.numpy()/.detach().cpu().numpy()/g' src/scFEA.py
print('Patchi (pandas append + torch cpu):')
!grep -nE "_append\(|detach\(\)\.cpu\(\)\.numpy\(\)" src/scFEA.py

# Preglej data/ folder — poisci glavni clovecji model.
print('\n=== scFEA data/ moduli in stoichiometry ===')
!ls data/ | grep -iE "module_gene|cmMat"

## 1. Poti in nalaganje RNA

scFEA potrebuje CSV: vrstice=geni (simboli), stolpci=celice.

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
data_parent_folder = '/content/drive/MyDrive/Diploma/data/processed'

with open(os.path.join(data_parent_folder, 'data_rna.pkl'), 'rb') as f:
    data_rna = pickle.load(f)           # CELICE x GENI (sparse ali dense)
with open(os.path.join(data_parent_folder, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
# gene_names.pkl shrani notebook 01 (cell-18). scFEA rabi gene SIMBOLE.
with open(os.path.join(data_parent_folder, 'gene_names.pkl'), 'rb') as f:
    gene_names = pickle.load(f)

print('data_rna:', data_rna.shape, type(data_rna))
print('data_labels:', data_labels.shape)
print('gene_names:', len(gene_names), '| primer:', list(gene_names[:5]))
assert len(gene_names) == data_rna.shape[1], 'gene_names se ne ujema s stevilom stolpcev data_rna!'

## 2. Diagnostika imen genov

So to gene simboli (CD8A) ali Ensembl ID (ENSG00000153563)?

In [ ]:
# So to gene simboli (CD8A) ali Ensembl ID (ENSG00000153563)?
sample = [str(g) for g in gene_names[:10]]
print('Prvih 10 genov:', sample)
is_ensembl = all(str(g).upper().startswith('ENSG') for g in gene_names[:50])
print('Videti kot Ensembl ID?', is_ensembl)
if is_ensembl:
    print('  -> POTREBNO: pretvori Ensembl ID v gene simbole (mygene/pybiomart) preden scFEA.')
else:
    print('  -> Videti kot gene simboli. scFEA ok.')

## 3. Vzorec za demo (proof-of-concept)

Vzemi MAJHEN vzorec (npr. 2000 celic enega pacienta), da izmerimo cas. Ne celoten dataset.

In [ ]:
from scipy.sparse import issparse

N_DEMO = 2000                  # demo velikost
# vzemi celice enega pacienta (P24) za reprezentativen vzorec
mask = (data_labels['Patient'].values == 24)
idx = np.where(mask)[0]
if len(idx) > N_DEMO:
    rng = np.random.RandomState(0)
    idx = rng.choice(idx, N_DEMO, replace=False)
print(f'Demo vzorec: {len(idx)} celic (pacient 24)')

X = data_rna[idx]
X = X.toarray() if issparse(X) else np.asarray(X)
print('X (log1p) min/max:', X.min(), X.max())

# KLJUC: data_rna je LOG1P (za TRIM). scFEA pricakuje SUROVE COUNTE (sam logira ce max>50).
# Log1p vhod (max~6) povzroci NaN (geneExprScale nestabilen). Obrni nazaj: expm1(log1p(x)) = x.
X = np.expm1(X)
X = np.rint(X)                 # surovi counti so cela stevila
print('X (surovi counti) min/max:', X.min(), X.max(), '-> scFEA bo sam log2 (max>50)')

# scFEA vhod: vrstice = gene simboli, stolpci = celice.
df_in = pd.DataFrame(X.T, index=[str(g) for g in gene_names],
                     columns=[f'cell_{i}' for i in range(X.shape[0])])
print('scFEA vhod (geni x celice):', df_in.shape)

os.makedirs('/content/scfea_input', exist_ok=True)
in_csv = '/content/scfea_input/demo_expr.csv'
df_in.to_csv(in_csv)
print('Shranjeno:', in_csv)

## 4. Poženi scFEA in izmeri čas

In [ ]:
%cd /content/scFEA
os.makedirs('/content/scfea_output', exist_ok=True)

# Izberi GLAVNI clovecji model. module_gene_m168.csv (168 modulov) potrebuje
# ujemajoco stoichiometry: cmMat z "m168" ali "171", NE specializirane (iron/glucose/c8...).
import glob as _glob, re
def pick(patterns, files, exclude=('mouse','iron','glucose','lactate','amino')):
    for pat in patterns:
        for p in files:
            b = os.path.basename(p).lower()
            if pat in b and not any(e in b for e in exclude):
                return os.path.basename(p)
    return None

mod_files  = _glob.glob('data/module_gene*.csv')
cmat_files = _glob.glob('data/cmMat*.csv')
module_file = pick(['m168','complete'], mod_files) or 'module_gene_m168.csv'
cmmat_file  = pick(['_171','m168','complete_171'], cmat_files) or 'cmMat_171.csv'
print('Uporabljam moduleGene:', module_file)
print('Uporabljam stoichiometry:', cmmat_file)
print('  (vsi cmMat na voljo:', [os.path.basename(p) for p in cmat_files], ')')

flux_out = '/content/scfea_output/flux.csv'
bal_out  = '/content/scfea_output/balance.csv'

t0 = time.time()
!python src/scFEA.py \
    --data_dir data \
    --input_dir /content/scfea_input \
    --test_file demo_expr.csv \
    --moduleGene_file {module_file} \
    --stoichiometry_matrix {cmmat_file} \
    --output_flux_file {flux_out} \
    --output_balance_file {bal_out} \
    --sc_imputation False
elapsed = time.time() - t0
print(f'\n=== scFEA cas za {len(idx)} celic: {elapsed:.1f} s ===')
print(f'Ekstrapolacija na 146776 celic: ~{elapsed/len(idx)*146776/60:.1f} min (linearno, groba ocena)')

## 5. Preglej izhod (flux matrika)

In [ ]:
import glob
# preberi eksplicitno doloceni flux.csv; fallback na karkoli v output mapi
cand = '/content/scfea_output/flux.csv'
if not os.path.exists(cand):
    others = glob.glob('/content/scfea_output/*.csv')
    print('flux.csv ni najden. Vse .csv v output:', others)
    cand = others[0] if others else None

if cand:
    flux = pd.read_csv(cand, index_col=0)
    print('Flux matrika:', flux.shape, '(pricakovano: celice x ~168 modulov)')
    print('Moduli (prvih 10):', list(flux.columns[:10]))
    print('\nStatistika flux vrednosti:')
    print('  min/max/mean:', flux.values.min(), flux.values.max(), flux.values.mean())
    print('  delez nicelnih:', (flux.values == 0).mean())
    print('\nPrvih 5 celic x 5 modulov:')
    print(flux.iloc[:5, :5])
else:
    print('Ni izhodne datoteke - preveri scFEA izpis (cell-11) za napake.')

In [ ]:
# === DIAGNOSTIKA NaN flux ===
# NaN (ne nicle) -> numericna napaka, pogosto: (a) premalo ujemanja gene-imen z moduli,
# (b) napacna normalizacija vhoda, (c) bug pri velikih datasetih.

# (a) Koliko nasih genov se ujema z geni v scFEA modulih?
import pandas as pd
mg = pd.read_csv(f'data/{module_file}', index_col=0)
# module_gene datoteka: zberi vsa gene imena iz nje
module_genes = set()
for col in mg.columns:
    for v in mg[col].dropna().astype(str):
        for g in v.replace(';',',').split(','):
            g = g.strip()
            if g and g.lower() != 'nan':
                module_genes.add(g)
our_genes = set(str(g) for g in gene_names)
overlap = our_genes & module_genes
print(f'scFEA modulnih genov: {len(module_genes)}')
print(f'nasih genov: {len(our_genes)}')
print(f'PRESEK: {len(overlap)}  ({100*len(overlap)/max(1,len(module_genes)):.1f}% modulnih genov pokritih)')
print('primer modulnih genov:', list(module_genes)[:10])
print('primer nasih genov:', list(our_genes)[:10])
if len(overlap) < 0.3 * len(module_genes):
    print('  -> SLABO UJEMANJE: verjetno NaN izvira iz neujemanja gene-imen (nomenklatura).')

# (b) Statistika vhoda — scFEA log-transformira ce vrednosti >30; nasi so log1p (majhni).
print(f'\nVhod X: min={X.min():.3f}, max={X.max():.3f}, mean={X.mean():.3f}')
print('  (scFEA auto-log ce max>30. Nas log1p ima max ~{:.1f} -> NE bo dvojno logiral.)'.format(X.max()))

## 6. Zaključek demo

Ce je demo uspel, imamo potrjeno:
- scFEA tece na nasih podatkih (format/imena genov ok)
- cas na vzorec -> ekstrapolacija na cel dataset (ali je <1-2h sprejemljivo?)
- flux izhod (celice x ~168 modulov) -> to bo 3. modaliteta v TRIM

Naslednji koraki (PO uspesnem demo):
1. Pridobi/pretvori gene simbole ce so Ensembl
2. Pozeni scFEA na CELOTNEM datasetu -> flux matrika (146776 x ~168)
3. Shrani data_flux.pkl (poravnan z data_rna po vrsticah)
4. Integracija: flux encoder/decoder kot 3. modaliteta v TRIM (notebook 03 flux varianta)

OPOZORILA za diplomo:
- scFEA fluksi so RELATIVNI, model-odvisni (ne absolutne hitrosti)
- benchmark scFEA vs Compass vs METAFlux ne obstaja -> trditi 'scFEA-ocenjeni fluksi'
